# Standalone 1-File Kaggle Submission — Final PyTorch Neural Network (`FraudMLP`)

**100% Self-Contained:** Requires zero external `.py` or `.json` files — upload this single `.ipynb` directly to Kaggle (GPU T4/P100 recommended)!  
**Model:** 4-Layer Tabular MLP (`256 -> 128 -> 64 -> 32`, `BatchNorm1d`, `ReLU`, `Dropout`), winning `EXP-NN-1A` configuration (`BCEWithLogitsLoss`, `pos_weight = 1.0`, `clip_val = 20.0`, AdamW + ReduceLROnPlateau + AUPRC Early Stopping).  
**Validation:** Leak-free Stratified 5-Fold CV with strict per-fold imputation, `log1p`, `RobustScaler`, and `OneHotEncoder`.  
**Output:** `/kaggle/working/submission.csv` (on Kaggle) or `submission_nn_final.csv` (locally).

---
## Section 1: Imports, Auto-Path Detection & Standalone Pipeline

In [ ]:
import copy
import math
import os
import random
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

LOCAL_SUBMISSION_PATH = "submission_nn_final.csv"
SEEDS = [42, 2024, 2026]  # Use [42] for faster 1-seed run; [42, 2024, 2026] for 3-seed bagging
HIDDEN_DIMS = (256, 128, 64, 32)
DROPOUTS = (0.30, 0.25, 0.20, 0.10)
CLIP_VAL = 20.0
BATCH_SIZE = 4096
MAX_EPOCHS = 22
PATIENCE = 5
LR = 2e-3
WEIGHT_DECAY = 1e-4

def resolve_data_paths() -> tuple[str, str, str, str]:
    """Auto-detect Kaggle competition input directory or fall back to local workspace."""
    is_kaggle = os.path.exists("/kaggle/input")
    if is_kaggle:
        found = {}
        for root, _, files in os.walk("/kaggle/input"):
            for fname in ("train.csv", "test.csv", "sample_submission.csv"):
                if fname in files and fname not in found:
                    found[fname] = os.path.join(root, fname)
        if len(found) == 3:
            return found["train.csv"], found["test.csv"], found["sample_submission.csv"], "/kaggle/working/submission.csv"
        raise FileNotFoundError(f"Missing competition CSVs under /kaggle/input. Found: {found}")
    base = "." if os.path.exists("train.csv") else ".."
    return (
        os.path.join(base, "train.csv"),
        os.path.join(base, "test.csv"),
        os.path.join(base, "sample_submission.csv"),
        LOCAL_SUBMISSION_PATH,
    )

SEED = 42
N_SPLITS = 5
ID_COL = "id"
TARGET_COL = "is_fraud"

CONT_COLS = [
    "amount_usd", "hours_since_last_txn", "txn_count_last_24h",
    "distance_from_home_km", "card_age_months", "customer_age",
    "account_balance_usd", "cvv_retry_count", "velocity_score",
    "merchant_risk_score", "prior_disputes",
]
TIME_COLS = ["time_of_day_hour", "day_of_week"]
CAT_COLS = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
RISK_FLAG_COLS = [
    "is_foreign_transaction", "is_new_merchant", "used_vpn",
    "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt",
]
ENGINEERED_COLS = [
    "amount_to_balance_ratio", "is_over_balance", "amount_per_txn_24h",
    "travel_speed_kmh", "is_brand_new_card", "has_cvv_retry",
    "hour_sin", "hour_cos", "is_night_txn", "is_weekend", "total_risk_flags",
]


def evaluate_auprc(y_true, y_pred_proba) -> float:
    y_true_arr = np.asarray(y_true, dtype=np.int64)
    y_prob_arr = np.asarray(y_pred_proba, dtype=np.float64)
    if np.isnan(y_prob_arr).any() or np.isinf(y_prob_arr).any():
        raise ValueError("Predicted probabilities contain NaN or Inf values.")
    return float(average_precision_score(y_true_arr, y_prob_arr))


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Deterministic row-level feature engineering (zero data leakage)."""
    out = df.copy()
    for flag_col in RISK_FLAG_COLS:
        out[flag_col] = out[flag_col].astype(float if out[flag_col].isnull().any() else int)

    out["amount_to_balance_ratio"] = out["amount_usd"] / (out["account_balance_usd"] + 1e-5)
    out["is_over_balance"] = (out["amount_usd"] > out["account_balance_usd"]).astype(int)
    out["amount_per_txn_24h"] = out["amount_usd"] / (out["txn_count_last_24h"] + 1)
    out["travel_speed_kmh"] = out["distance_from_home_km"] / (out["hours_since_last_txn"] + 0.1)
    out["is_brand_new_card"] = (out["card_age_months"] <= 1).astype(int)
    out["has_cvv_retry"] = (out["cvv_retry_count"] > 0).astype(int)
    out["hour_sin"] = np.sin(2.0 * np.pi * out["time_of_day_hour"] / 24.0)
    out["hour_cos"] = np.cos(2.0 * np.pi * out["time_of_day_hour"] / 24.0)
    out["is_night_txn"] = out["time_of_day_hour"].between(0, 5).astype(int)

    dow = out["day_of_week"]
    if pd.api.types.is_numeric_dtype(dow):
        dow_valid = dow.dropna()
        if len(dow_valid) > 0 and dow_valid.min() >= 1 and dow_valid.max() == 7:
            out["is_weekend"] = dow.isin([6, 7]).astype(int)
        else:
            out["is_weekend"] = dow.isin([5, 6]).astype(int)
    else:
        weekend_strings = {"saturday", "sunday", "sat", "sun"}
        out["is_weekend"] = dow.astype(str).str.strip().str.lower().isin(weekend_strings).astype(int)

    out["total_risk_flags"] = out[RISK_FLAG_COLS].sum(axis=1)
    return out

NN_NUM_COLS = CONT_COLS + TIME_COLS + [
    "amount_to_balance_ratio", "amount_per_txn_24h", "travel_speed_kmh",
    "hour_sin", "hour_cos", "total_risk_flags",
]
NN_SKEW_COLS = ["amount_usd", "account_balance_usd", "distance_from_home_km", "hours_since_last_txn", "travel_speed_kmh"]
NN_BIN_COLS = RISK_FLAG_COLS + ["is_over_balance", "is_brand_new_card", "has_cvv_retry", "is_night_txn", "is_weekend"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH, SUBMISSION_PATH = resolve_data_paths()
print(f"Device         : {device}")
print(f"Train CSV      : {TRAIN_PATH}")
print(f"Test CSV       : {TEST_PATH}")
print(f"Submission Out : {SUBMISSION_PATH}")
print(f"Seeds          : {SEEDS} | Hidden Dims: {HIDDEN_DIMS}")

---
## Section 2: PyTorch `FraudMLP` & Leak-Free 5-Fold Training / Inference

In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


class FraudMLP(nn.Module):
    def __init__(self, in_features: int, hidden_dims=HIDDEN_DIMS, dropouts=DROPOUTS, init_pos_prob: float | None = None):
        super().__init__()
        layers = []
        prev_dim = in_features
        for h_dim, drop in zip(hidden_dims, dropouts):
            lin = nn.Linear(prev_dim, h_dim)
            nn.init.kaiming_normal_(lin.weight, nonlinearity="relu")
            nn.init.zeros_(lin.bias)
            layers.extend([lin, nn.BatchNorm1d(h_dim), nn.ReLU(), nn.Dropout(p=drop)])
            prev_dim = h_dim
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(prev_dim, 1)
        nn.init.xavier_normal_(self.head.weight)
        if init_pos_prob is not None and 0.0 < init_pos_prob < 1.0:
            nn.init.constant_(self.head.bias, math.log(init_pos_prob / (1.0 - init_pos_prob)))
        else:
            nn.init.zeros_(self.head.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x)).squeeze(-1)


@torch.no_grad()
def predict_proba_tensor(model: nn.Module, X_tensor: torch.Tensor, batch_size: int = 16384) -> np.ndarray:
    model.eval()
    n = X_tensor.shape[0]
    preds = np.empty(n, dtype=np.float64)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        preds[start:end] = torch.sigmoid(model(X_tensor[start:end])).detach().cpu().numpy()
    return preds


def _prep_cat_matrix(df: pd.DataFrame, cols: list[str]) -> np.ndarray:
    arr = df[cols].astype(object).to_numpy(copy=True)
    arr[pd.isna(arr)] = np.nan
    return arr


def _normalize_dow(df: pd.DataFrame) -> pd.DataFrame:
    if pd.api.types.is_numeric_dtype(df["day_of_week"]):
        return df
    out = df.copy()
    dow_map = {"monday": 0, "tuesday": 1, "wednesday": 2, "thursday": 3, "friday": 4, "saturday": 5, "sunday": 6}
    out["day_of_week"] = out["day_of_week"].astype(str).str.lower().map(dow_map)
    return out


def transform_fold(train_fe: pd.DataFrame, test_fe: pd.DataFrame, tr_idx: np.ndarray, va_idx: np.ndarray):
    skew_idx = [NN_NUM_COLS.index(c) for c in NN_SKEW_COLS]

    num_imp = SimpleImputer(strategy="median")
    X_tr_n = num_imp.fit_transform(train_fe.iloc[tr_idx][NN_NUM_COLS].to_numpy(dtype=np.float64))
    X_va_n = num_imp.transform(train_fe.iloc[va_idx][NN_NUM_COLS].to_numpy(dtype=np.float64))
    X_te_n = num_imp.transform(test_fe[NN_NUM_COLS].to_numpy(dtype=np.float64))

    for arr in (X_tr_n, X_va_n, X_te_n):
        arr[:, skew_idx] = np.log1p(np.clip(arr[:, skew_idx], 0.0, None))

    scaler = RobustScaler()
    X_tr_n = scaler.fit_transform(X_tr_n)
    X_va_n = scaler.transform(X_va_n)
    X_te_n = scaler.transform(X_te_n)

    bin_imp = SimpleImputer(strategy="most_frequent")
    X_tr_b = bin_imp.fit_transform(train_fe.iloc[tr_idx][NN_BIN_COLS].to_numpy(dtype=np.float64))
    X_va_b = bin_imp.transform(train_fe.iloc[va_idx][NN_BIN_COLS].to_numpy(dtype=np.float64))
    X_te_b = bin_imp.transform(test_fe[NN_BIN_COLS].to_numpy(dtype=np.float64))

    cat_imp = SimpleImputer(strategy="most_frequent")
    X_tr_ci = cat_imp.fit_transform(_prep_cat_matrix(train_fe.iloc[tr_idx], CAT_COLS))
    X_va_ci = cat_imp.transform(_prep_cat_matrix(train_fe.iloc[va_idx], CAT_COLS))
    X_te_ci = cat_imp.transform(_prep_cat_matrix(test_fe, CAT_COLS))

    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    X_tr_c = ohe.fit_transform(X_tr_ci)
    X_va_c = ohe.transform(X_va_ci)
    X_te_c = ohe.transform(X_te_ci)

    X_tr = np.clip(np.hstack([X_tr_n, X_tr_b, X_tr_c]), -CLIP_VAL, CLIP_VAL).astype(np.float32)
    X_va = np.clip(np.hstack([X_va_n, X_va_b, X_va_c]), -CLIP_VAL, CLIP_VAL).astype(np.float32)
    X_te = np.clip(np.hstack([X_te_n, X_te_b, X_te_c]), -CLIP_VAL, CLIP_VAL).astype(np.float32)
    return X_tr, X_va, X_te


def train_and_predict_seed(X_tr, y_tr, X_va, y_va, X_te, seed: int):
    set_seed(seed)
    prior = float(np.mean(y_tr))
    model = FraudMLP(X_tr.shape[1], HIDDEN_DIMS, DROPOUTS, init_pos_prob=prior).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    X_tr_t = torch.from_numpy(X_tr).to(device)
    y_tr_t = torch.from_numpy(y_tr.astype(np.float32)).to(device)
    X_va_t = torch.from_numpy(X_va).to(device)
    X_te_t = torch.from_numpy(X_te).to(device)

    n_train = X_tr_t.shape[0]
    best_auprc, best_ep, no_improve = -1.0, 0, 0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n_train, device=device)
        for start in range(0, n_train, BATCH_SIZE):
            end = min(start + BATCH_SIZE, n_train)
            if end - start < 2:
                continue
            idx = perm[start:end]
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X_tr_t[idx]), y_tr_t[idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        val_probs = predict_proba_tensor(model, X_va_t)
        val_auprc = evaluate_auprc(y_va, val_probs)
        scheduler.step(val_auprc)

        if val_auprc > best_auprc + 1e-6:
            best_auprc, best_ep, no_improve = val_auprc, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return predict_proba_tensor(model, X_va_t), predict_proba_tensor(model, X_te_t), best_ep


t0 = time.perf_counter()
train_fe = _normalize_dow(build_features(pd.read_csv(TRAIN_PATH)))
test_fe = _normalize_dow(build_features(pd.read_csv(TEST_PATH)))
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
assert np.array_equal(test_fe[ID_COL].to_numpy(), sample_sub[ID_COL].to_numpy()), "Test ID mismatch!"

y_all = train_fe[TARGET_COL].astype(np.int64).to_numpy()
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_preds = np.zeros(len(train_fe), dtype=np.float64)
test_preds = np.zeros(len(test_fe), dtype=np.float64)
fold_auprcs = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_fe, y_all)):
    X_tr, X_va, X_te = transform_fold(train_fe, test_fe, tr_idx, va_idx)
    if fold == 0:
        print(f"Preprocessed NN input dimension: {X_tr.shape[1]}")

    fold_val = np.zeros(len(va_idx), dtype=np.float64)
    fold_test = np.zeros(len(test_fe), dtype=np.float64)
    eps = []

    for seed in SEEDS:
        vp, tp, best_ep = train_and_predict_seed(X_tr, y_all[tr_idx], X_va, y_all[va_idx], X_te, seed=int(seed) + fold)
        fold_val += vp / len(SEEDS)
        fold_test += tp / len(SEEDS)
        eps.append(best_ep)

    oof_preds[va_idx] = fold_val
    test_preds += fold_test / N_SPLITS
    f_auprc = evaluate_auprc(y_all[va_idx], fold_val)
    fold_auprcs.append(f_auprc)
    print(f"Fold {fold} | Bagged Val AUPRC: {f_auprc:.6f} | Best Epochs: {eps}")

np.save("oof_nn.npy", oof_preds)
np.save("test_nn.npy", test_preds)

oof_auprc = evaluate_auprc(y_all, oof_preds)
oof_roc = float(roc_auc_score(y_all, oof_preds))
print(f"\nNN OOF AUPRC: {oof_auprc:.6f} | Mean Fold: {np.mean(fold_auprcs):.6f} +/- {np.std(fold_auprcs):.6f} | ROC-AUC: {oof_roc:.6f} | Time: {time.perf_counter()-t0:.1f}s")

---
## Section 3: Diagnostics & Export Validated Kaggle Submission

In [ ]:
prec, rec, _ = precision_recall_curve(y_all, oof_preds)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].plot(rec, prec, color="#2ca02c", lw=2.2, label=f"FraudMLP (AUPRC={oof_auprc:.6f})")
axes[0].axhline(y_all.mean(), color="#d62728", ls="--", lw=1.2, label=f"Prior ({y_all.mean():.4f})")
axes[0].set_title("NN OOF Precision-Recall Curve", fontweight="bold")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision"); axes[0].legend()

bins = np.linspace(0.0, 1.0, 51)
axes[1].hist(oof_preds, bins=bins, density=True, alpha=0.55, color="#2ca02c", label="OOF (Train)")
axes[1].hist(test_preds, bins=bins, density=True, alpha=0.45, color="#ff7f0e", label="Test (Kaggle)")
axes[1].set_yscale("log")
axes[1].set_title("NN OOF vs. Test Fraud Probability Distribution", fontweight="bold")
axes[1].set_xlabel("Predicted Probability"); axes[1].set_ylabel("Density (log)"); axes[1].legend()
plt.tight_layout(); plt.show()

submission = pd.DataFrame({ID_COL: sample_sub[ID_COL].astype(np.int64), TARGET_COL: test_preds.astype(np.float64)})
submission.to_csv(SUBMISSION_PATH, index=False)

sub_check = pd.read_csv(SUBMISSION_PATH)
assert sub_check.shape == sample_sub.shape, f"Shape mismatch: {sub_check.shape} != {sample_sub.shape}"
assert list(sub_check.columns) == list(sample_sub.columns), f"Columns mismatch: {list(sub_check.columns)}"
assert np.array_equal(sub_check[ID_COL].to_numpy(), sample_sub[ID_COL].to_numpy()), "ID order mismatch!"
assert sub_check[TARGET_COL].isna().sum() == 0 and np.isfinite(sub_check[TARGET_COL].to_numpy()).all(), "NaN/Inf detected!"
assert (sub_check[TARGET_COL] >= 0.0).all() and (sub_check[TARGET_COL] <= 1.0).all(), "Probabilities out of [0, 1]!"
assert sub_check[TARGET_COL].std() > 1e-4, "Predictions appear constant!"

print(f"ALL KAGGLE NN SUBMISSION CHECKS PASSED -> {os.path.abspath(SUBMISSION_PATH)} ({os.path.getsize(SUBMISSION_PATH)/1e6:.2f} MB)")
display(sub_check.head(10))